# Naive Bayes — classification juridique arabe

Pipeline reproductible utilisant les modules de `src/` : chargement UTF-8, prétraitement arabe, Naive Bayes maison, validation croisée, métriques et prédiction sauvegardée.

Exécuter les cellules dans l'ordre avec le noyau du projet (Python 3.11/3.12 et dépendances `pip install -e ".[dev]"`). Aucune installation ni aucun téléchargement ne sont lancés par ce notebook. Les sorties sont volontairement vides tant que le pipeline n'a pas été exécuté.

Le vocabulaire est appris dans chaque pli. Le jeu de test reste exclu du choix des hyperparamètres. Les textes originaux restent inchangés.


In [1]:
import hashlib
import json
import logging
import sys
from collections import Counter
from dataclasses import asdict
from datetime import datetime, timezone
from functools import partial
from html import escape
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import numpy as np
from IPython.display import HTML, display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

# Fonctionne depuis la racine du projet ou son dossier notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Ouvrir le notebook depuis le projet ArabicLegalTextMining.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification.evaluation import classification_metrics, stratified_cross_validation
from src.classification.naive_bayes import MultinomialNaiveBayes
from src.classification.predictor import NaiveBayesPredictor
from src.common.document_loader import CLASSES, load_documents
from src.common.arabic_normalizer import NormalizationConfig
from src.lexical_search.preprocessing import DEFAULT_STOPWORDS, PreprocessingConfig, preprocess_text

logging.basicConfig(level=logging.WARNING, format="%(levelname)s : %(message)s")
print("Projet :", PROJECT_ROOT)


Projet : c:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining


## 1. Paramètres

Les comptages bruts sont utilisés, sans TF-IDF ni sélection Chi². Les choix de normalisation sont fixes pour cette expérience. En cas d'effectifs insuffisants, le notebook s'arrête plutôt que de modifier silencieusement le protocole.


In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "naive_bayes"
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
ALPHAS = (0.5, 1.0, 2.0)
STRICT_LOADING = True
CONFIG = PreprocessingConfig(
    normalization=NormalizationConfig(),
    stemming=True,
    keep_numbers=True,
)
STOPWORDS = tuple(sorted(DEFAULT_STOPWORDS))

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

def show_metrics(report):
    show_table(
        ["Classe", "Précision", "Rappel", "F1", "Support"],
        [[row["label"], f'{row["precision"]:.4f}', f'{row["recall"]:.4f}',
          f'{row["f1"]:.4f}', row["support"]] for row in report["per_class"]],
    )
    print("Accuracy :", round(report["accuracy"], 4))
    print("Moyennes macro :", report["macro_avg"])
    print("Moyennes pondérées :", report["weighted_avg"])
    print("Confusion : lignes = réelles, colonnes = prédites")
    show_table(
        ["Réelle / prédite", *report["labels"]],
        [[label, *row] for label, row in zip(report["labels"], report["confusion_matrix"])],
    )


## 2. Chargement et contrôle du corpus

Le mode strict signale immédiatement un fichier illisible ou mal encodé. Les documents vides sont conservés : le classifieur pourra les traiter avec ses a priori. Le diagnostic de doublons porte sur le texte original exact ; des duplications ou documents issus d'une même affaire nécessitent un découpage par groupe avant de tirer des conclusions sur la généralisation.


In [3]:
documents = load_documents(DATA_DIR, strict=STRICT_LOADING)
if not documents:
    raise ValueError("Le corpus est vide.")
counts = Counter(document["class"] for document in documents)
if any(counts[label] < 2 for label in CLASSES):
    raise ValueError("Chaque classe doit contenir au moins deux documents.")
if len({document["id"] for document in documents}) != len(documents):
    raise ValueError("Les identifiants des documents doivent être uniques.")

texts = np.asarray([document["text"] for document in documents], dtype=object)
labels = np.asarray([document["class"] for document in documents])
document_ids = np.asarray([document["id"] for document in documents])
show_table(["Classe", "Documents"], sorted(counts.items()))
print("Documents vides :", sum(not text.strip() for text in texts))
print("Textes identiques supplémentaires :", len(texts) - len(set(texts)))

sample = documents[0]
print("Exemple :", sample["id"])
print("Texte original (extrait) :", sample["text"][:200])
print("Tokens :", preprocess_text(sample["text"], CONFIG, stopwords=STOPWORDS)[:30])


Classe,Documents
family_inheritance,1883
real_estate,365


Documents vides : 7
Textes identiques supplémentaires : 7
Exemple : 8646d576-30a8-5bf2-a118-e22e9b7e1643
Texte original (extrait) : قرار محكمة النقض
رقم 270
الصادر بتاريخ 13 يونيو 2023
في الملف الشرعي رقم 2022/2/2/326
شیاع - طلب فرز نصيب مع تعويض عن الحرمان من الاستغلال – موجباته.
بمقتضى المادة 235 من مدونة الحقوق العينية، فإن كل 
Tokens : ['قرار', 'محكم', 'نقض', 'رقم', '270', 'صادر', 'تاريخ', '13', 'يون', '2023', 'ملف', 'شرع', 'رقم', '2022', '2', '2', '326', 'شیاع', 'طلب', 'فرز', 'نصيب', 'تعويض', 'حرم', 'استغلال', 'موجب', 'مقتضى', 'ماد', '235', 'مدون', 'حقوق']


## 3. Jeu de test indépendant

Le découpage est stratifié et reproductible. La validation croisée et le choix d'alpha utilisent uniquement la partie entraînement.


In [4]:
train_indices, test_indices = train_test_split(
    np.arange(len(documents)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)
train_texts, test_texts = texts[train_indices], texts[test_indices]
train_labels, test_labels = labels[train_indices], labels[test_indices]
if min(Counter(train_labels).values()) < N_SPLITS:
    raise ValueError("Effectifs insuffisants pour N_SPLITS : réduire ce paramètre explicitement.")
assert not set(train_indices) & set(test_indices)

show_table(
    ["Classe", "Entraînement", "Test"],
    [[label, int(np.sum(train_labels == label)), int(np.sum(test_labels == label))]
     for label in sorted(counts)],
)


Classe,Entraînement,Test
family_inheritance,1506,377
real_estate,292,73


## 4. Validation croisée et sélection du lissage

La fabrique renvoie un nouveau CountVectorizer par pli. Le prétraitement est celui du projet ; aucun vocabulaire n'est calculé sur le test. Le modèle Naive Bayes et les métriques proviennent des modules Python.

Le meilleur alpha maximise la moyenne du F1 macro par pli. En cas d'égalité, le plus petit alpha est retenu. L'écart-type affiché décrit la dispersion entre plis, pas un intervalle de confiance.


In [5]:
def make_vectorizer():
    return CountVectorizer(
        analyzer=partial(preprocess_text, config=CONFIG, stopwords=STOPWORDS),
        lowercase=False,
        token_pattern=None,
        dtype=np.float64,
    )

cv_reports = {}
for alpha in ALPHAS:
    cv_reports[alpha] = stratified_cross_validation(
        train_texts,
        train_labels,
        n_splits=N_SPLITS,
        alpha=alpha,
        random_state=RANDOM_STATE,
        vectorizer_factory=make_vectorizer,
    )

best_alpha = min(
    cv_reports,
    key=lambda alpha: (-cv_reports[alpha]["mean_scores"]["f1"], alpha),
)
show_table(
    ["Alpha", "Accuracy moyenne", "Précision macro", "Rappel macro", "F1 macro", "Écart-type F1"],
    [[alpha,
      f'{report["mean_scores"]["accuracy"]:.4f}',
      f'{report["mean_scores"]["precision"]:.4f}',
      f'{report["mean_scores"]["recall"]:.4f}',
      f'{report["mean_scores"]["f1"]:.4f}',
      f'{report["std_scores"]["f1"]:.4f}']
     for alpha, report in cv_reports.items()],
)
print("Alpha retenu :", best_alpha)
print("Rapport agrégé des prédictions hors apprentissage :")
show_metrics(cv_reports[best_alpha]["metrics"])


Alpha,Accuracy moyenne,Précision macro,Rappel macro,F1 macro,Écart-type F1
0.5,0.9672,0.9214,0.9694,0.9430,0.0054
1.0,0.9694,0.9266,0.9707,0.9466,0.0085
2.0,0.9800,0.9541,0.9757,0.9641,0.0072


Alpha retenu : 2.0
Rapport agrégé des prédictions hors apprentissage :


Classe,Précision,Rappel,F1,Support
family_inheritance,0.9940,0.9821,0.9880,1506
real_estate,0.9129,0.9692,0.9402,292


Accuracy : 0.98
Moyennes macro : {'precision': 0.9534274193548387, 'recall': 0.9756248976695956, 'f1': 0.9640876437259902}
Moyennes pondérées : {'precision': 0.9807891384692669, 'recall': 0.9799777530589544, 'f1': 0.9802169018616245}
Confusion : lignes = réelles, colonnes = prédites


Réelle / prédite,family_inheritance,real_estate
family_inheritance,1479,27
real_estate,9,283


## 5. Entraînement et évaluation finale

Le vocabulaire final est appris sur toute la partition d'entraînement. Le jeu de test est transformé sans ajustement. Ses résultats ne doivent pas servir à retoucher alpha ou le prétraitement ; sinon il ne s'agit plus d'une évaluation indépendante.


In [6]:
vectorizer = make_vectorizer()
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)
model = MultinomialNaiveBayes(alpha=best_alpha).fit(X_train, train_labels)
test_predictions = model.predict(X_test)
test_report = classification_metrics(test_labels, test_predictions, labels=sorted(counts))
print("Matrice entraînement :", X_train.shape, "| Matrice test :", X_test.shape)
print("Documents test sans terme connu :", int(np.sum(X_test.getnnz(axis=1) == 0)))
show_metrics(test_report)

# Naive baseline : classe majoritaire déterminée sur l'entraînement uniquement.
train_counts = Counter(train_labels)
majority_class = min(train_counts, key=lambda label: (-train_counts[label], label))
baseline_report = classification_metrics(
    test_labels, [majority_class] * len(test_labels), labels=sorted(counts)
)
show_table(
    ["Méthode", "Accuracy test", "F1 macro test"],
    [["Naive Bayes", test_report["accuracy"], test_report["macro_avg"]["f1"]],
     ["Classe majoritaire", baseline_report["accuracy"], baseline_report["macro_avg"]["f1"]]],
)


Matrice entraînement : (1798, 18997) | Matrice test : (450, 18997)
Documents test sans terme connu : 1


Classe,Précision,Rappel,F1,Support
family_inheritance,0.9946,0.9761,0.9853,377
real_estate,0.8875,0.9726,0.9281,73


Accuracy : 0.9756
Moyennes macro : {'precision': 0.9410472972972973, 'recall': 0.9743650303404673, 'f1': 0.9566895031104812}
Moyennes pondérées : {'precision': 0.9772214714714715, 'recall': 0.9755555555555555, 'f1': 0.976000209990288}
Confusion : lignes = réelles, colonnes = prédites


Réelle / prédite,family_inheritance,real_estate
family_inheritance,368,9
real_estate,2,71


Méthode,Accuracy test,F1 macro test
Naive Bayes,0.9755555555555555,0.9566895031104812
Classe majoritaire,0.8377777777777777,0.4558645707376058


## 6. Prédicteur texte et examen des erreurs

Le prédicteur réutilise exactement les colonnes du vectoriseur et les paramètres de prétraitement. Le contrôle ci-dessous vérifie l'équivalence entre la prédiction depuis le texte brut et celle depuis les comptages.


In [7]:
predictor = NaiveBayesPredictor(
    model, vectorizer.vocabulary_, config=CONFIG, stopwords=STOPWORDS
)
assert predictor.predict_many(test_texts) == test_predictions.tolist()
test_records = [
    {"id": str(document_ids[index]), "actual": str(actual), "predicted": str(predicted)}
    for index, actual, predicted in zip(test_indices, test_labels, test_predictions)
]
errors = [record for record in test_records if record["actual"] != record["predicted"]]
print("Erreurs test :", len(errors), "/", len(test_records))
show_table(
    ["Document", "Classe réelle", "Classe prédite"],
    [[record["id"], record["actual"], record["predicted"]] for record in errors[:10]],
)
example_text = "عقد ملكية العقار"
print("Exemple de prédiction :", predictor.predict(example_text))
print("Probabilités :", predictor.predict_proba(example_text))


Erreurs test : 11 / 450


Document,Classe réelle,Classe prédite
0972816a-447d-50ac-b87f-eb65d7c7a53c,family_inheritance,real_estate
913d6c70-c2a3-52ed-a4c8-a84d466bea72,family_inheritance,real_estate
e90f7ac3-c92a-5d14-9a8c-fd56d11621f9,family_inheritance,real_estate
4ad88c95-9383-5e46-aee9-6e916a82db54,real_estate,family_inheritance
c85249cb-f692-5f65-9a9d-326b7c61dce7,family_inheritance,real_estate
293bd995-44d4-530a-a816-7c58b562c95c,family_inheritance,real_estate
4fc0aef9-0e09-515d-987f-e06a9488dadd,family_inheritance,real_estate
ff3e2cb6-5aa4-5c7e-838e-3edd2014a2c5,family_inheritance,real_estate
3cce4b3e-4ada-563a-a158-de7963125110,family_inheritance,real_estate
e8496fbd-bff4-578f-8204-08609c954410,family_inheritance,real_estate


Exemple de prédiction : real_estate
Probabilités : {'family_inheritance': 0.3941352545798421, 'real_estate': 0.6058647454201579}


## 7. Export et vérification du rechargement

Chaque exécution écrit dans un nouveau dossier sous `artifacts/naive_bayes/`. Le modèle exporté est **celui évalué**, entraîné uniquement sur la partition d'entraînement : les scores du test lui correspondent.

Le dossier contient le prédicteur autonome, le vocabulaire, les rapports de validation croisée et de test, les identifiants des partitions et un manifeste (paramètres, versions et empreinte du corpus). Les textes bruts ne sont pas exportés. Le manifeste est écrit après vérification du rechargement. Charger uniquement des fichiers joblib de confiance.


In [8]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)

def write_json(filename, payload):
    (run_dir / filename).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )

predictor_path = run_dir / "predictor.joblib"
predictor.save(predictor_path)
restored = NaiveBayesPredictor.load(predictor_path)
assert restored.predict_many(test_texts) == test_predictions.tolist()
for text in test_texts[:5]:
    np.testing.assert_allclose(
        list(restored.predict_proba(text).values()),
        list(predictor.predict_proba(text).values()),
    )

write_json("vocabulary.json", predictor.vocabulary_)
write_json("cross_validation.json", {str(alpha): report for alpha, report in cv_reports.items()})
write_json("test_metrics.json", test_report)
write_json("baseline_metrics.json", baseline_report)
write_json("test_predictions.json", test_records)
write_json("split.json", {
    "train_document_ids": document_ids[train_indices].tolist(),
    "test_document_ids": document_ids[test_indices].tolist(),
    "cv_index_reference": "Les indices CV désignent les positions dans train_document_ids.",
})

corpus_digest = hashlib.sha256()
for document in sorted(documents, key=lambda document: document["id"]):
    corpus_digest.update(json.dumps(
        [document["id"], document["class"], document["text"]], ensure_ascii=False
    ).encode("utf-8"))
    corpus_digest.update(b"\n")

write_json("manifest.json", {
    "version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": "MultinomialNaiveBayes",
    "training_scope": "train_partition_only",
    "best_alpha": best_alpha,
    "selection_metric": "mean_cv_macro_f1",
    "alpha_candidates": list(ALPHAS),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "n_splits": N_SPLITS,
    "strict_loading": STRICT_LOADING,
    "data_dir": str(DATA_DIR),
    "corpus_sha256": corpus_digest.hexdigest(),
    "n_documents": len(documents),
    "class_counts": dict(counts),
    "vocabulary_size": len(predictor.vocabulary_),
    "preprocessing": asdict(CONFIG),
    "stopwords": list(STOPWORDS),
    "python_version": sys.version,
    "packages": {name: version(name) for name in ("numpy", "scipy", "scikit-learn", "nltk", "joblib")},
    "files": sorted(path.name for path in run_dir.iterdir()),
})
print("Export vérifié :", run_dir)
print("Chargement : NaiveBayesPredictor.load(predictor_path)")


Export vérifié : c:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining\artifacts\naive_bayes\20261115T184543Z-ccb58889
Chargement : NaiveBayesPredictor.load(predictor_path)


## Références techniques

Les métriques, le classifieur et la persistance sont implémentés dans les modules du projet. Scikit-learn fournit uniquement la vectorisation et les découpages :
- [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
